In [ ]:
import os
import pickle
import matplotlib.pyplot as plt
import numpy as np

methods = ["wycoff", "vellanky", "kundu", "vien", "shilton"]
for target_fn in ["sinc"]:
    for lenghtscale in [0.1]:
        plt.figure(figsize=(14, 5))
        for i, method in enumerate(methods):
            save_dir = f"results/{method}/{target_fn}/"
            # load all pkl files in the directory
            results = [
                pickle.load(open(os.path.join(save_dir, f), "rb"))
                for f in os.listdir(save_dir)
                if f.endswith(".pkl")
                if f.startswith(f"lengthscale_{lenghtscale}")
            ]
            ys = np.array([r["observation_values"] for r in results])
            t_fit = np.array([r["surrogate_fit_time"] for r in results])
            t_acq = np.array([r["acquisition_time"] for r in results])
            t_targ = np.array([r["target_evaluation_time"] for r in results])

            plt.subplot(121)
            plt.title(f"target function")
            runs, max_acquisitions = ys.shape
            y_best = np.minimum.accumulate(ys, axis=-1)
            x = np.arange(max_acquisitions)
            # ci on median
            qu = np.clip(0.5 + 1.96 * np.sqrt(0.25 / runs), 0, 1)
            ql = np.clip(0.5 - 1.96 * np.sqrt(0.25 / runs), 0, 1)
            ul = np.quantile(y_best, qu, axis=0)
            ll = np.quantile(y_best, ql, axis=0)
            plt.plot(x, np.median(y_best, axis=0), label=f"{method}")
            plt.fill_between(x, ll, ul, alpha=0.2, color=plt.gca().lines[-1].get_color())
            plt.ylabel("target")
            plt.xlabel("acquisitions")
            plt.yscale("log")
            plt.grid(True)
            plt.legend()

            plt.subplot(143)
            plt.title(f"surrogate fit time")
            plt.boxplot(t_fit, positions=[i], patch_artist=True)
            plt.xticks(range(len(methods)), methods, rotation=45)
            plt.ylabel(f"Time (seconds)")
            plt.ylim(0, None)
            plt.grid(True)

            plt.subplot(144)
            plt.title(f"candidate search time")
            plt.boxplot(t_acq, positions=[i], patch_artist=True)
            plt.xticks(range(len(methods)), methods, rotation=45)
            plt.ylabel(f"Time (seconds)")
            plt.ylim(0, None)
            plt.grid(True)
        plt.tight_layout()
        plt.show()